# HEILO Faísca Code 0.3: aprender "pedido → código que funciona"

**Onde a Code 0.2 parou:** ela escreve Python válido (93%), mas acerta a lógica em 0 de 20 na primeira tentativa. Ela leu muito código solto, mas nunca viu **pedido e solução certa** juntos.

**O que a 0.3 faz:** continua da Code 0.2 (checkpoint no seu Drive) e treina ~90 minutos misturando:
- **70%** do código público que ela já lia (para não esquecer);
- **30%** de **296 funções verificadas**: pedido em português ou inglês, solução certa e **testes executados de verdade** (só entra o que passa). Converter unidades, listas, textos, arquivos, contas de usinagem (rpm, avanço)…

**Régua honesta:** nenhuma dessas funções é igual às 20 da prova, nem "irmã" direta delas (ímpar × par, minúsculas × maiúsculas…). Se ela acertar mais na prova, é porque **generalizou**.

**Antes:** GPU T4 e o secret `GITHUB_TOKEN`. Depois, `Executar tudo`. Leva cerca de 2 horas. Pode rodar junto com outro Colab.

In [ ]:
# 1) Configuração
REPO = "hugolindo12/HELIO"
BRANCH = "main"
NOME_VERSAO = "v0.3"
MINUTOS = 90
LOTE = 32
import torch, time, os, sys, json, shutil, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative a GPU!")

In [ ]:
# 2) Drive (checkpoint da Code 0.2)
from google.colab import drive
drive.mount("/content/drive")
PASTA = "/content/drive/MyDrive/heilo_seed_code"
assert os.path.exists(f"{PASTA}/tokenizer.json") and os.path.exists(f"{PASTA}/pretreino.pt"), \
    "Não achei o tokenizer/checkpoint da Code no Drive (pasta heilo_seed_code)."

In [ ]:
# 3) Código da HEILO (pasta própria)
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
URL = f"https://{TOKEN}@github.com/{REPO}.git"
BASE = "/content/code_run"
if not os.path.exists(f"{BASE}/heilo"):
    os.makedirs(BASE, exist_ok=True)
    subprocess.run(["git", "clone", "-b", BRANCH, URL, f"{BASE}/heilo"], check=True)
else:
    subprocess.run(f"cd {BASE}/heilo && git stash -u -q; git pull -q --rebase", shell=True)
sys.path.insert(0, BASE)
!pip -q install datasets tokenizers
from heilo.training import seed_code, pretrain, tarefas_codigo
from heilo.models.seed.tokenizer import BPETokenizer
from pathlib import Path
tok = BPETokenizer.from_json(Path(PASTA, "tokenizer.json").read_text(encoding="utf-8"))

In [ ]:
# 4) Corpus: código público (copiado para o disco rápido) + funções verificadas (30%)
fonte = Path(PASTA) / "corpus_grande"
if not (fonte / "meta.json").exists():
    fonte = Path(PASTA) / "corpus"
local = Path("/content/code_corpus")
if not (local / "meta.json").exists():
    t0 = time.time(); shutil.copytree(fonte, local, dirs_exist_ok=True)
    print(f"corpus de código copiado em {(time.time()-t0)/60:.1f} min")
tarefas = Path("/content/tarefas_verificadas")
meta = tarefas_codigo.preparar_corpus_tarefas(tok, tarefas, repeticoes=20, peso_fixo=0.3)
print("funções verificadas:", meta["relatorio"], "|", meta["tokens_treino"], "tokens")

In [ ]:
# 5) Continua da Code 0.2 (guarda uma cópia dela antes) — 90 min pelo relógio
ck = Path(PASTA) / "pretreino.pt"
copia = Path(PASTA) / "pretreino_v0.2.pt"
if not copia.exists():
    shutil.copy(ck, copia)
res = pretrain.pretreinar(seed_code.CONFIG_CODE, tok, [local, tarefas], ck, passos=10**9, lote=LOTE,
                          lr=2e-4, lr_min=2e-5, aquecimento=200, minutos_alvo=MINUTOS,
                          tempo_max_min=MINUTOS + 20, avaliar_cada=250, salvar_cada=500)
modelo = res["modelo"]
print("treino:", res["passo"], "passos |", res["hist"][-1] if res["hist"] else "")

In [ ]:
# 6) Prova REAL: completa as 20 funções e EXECUTA os testes
av = seed_code.avaliar_codigo(modelo)
resumo = {k: v for k, v in av.items() if k != "detalhes"}
print("Code 0.3:", resumo)
print("Code 0.2: {'pass@1': 0.0, 'pass@6': 0.05, 'codigo_valido': 0.93}")
for d in av["detalhes"]:
    if d["passou_greedy"] or d.get("passou_em_6"):
        print("-" * 40, "PASSOU:", d["problema"]); print(d["codigo"])

In [ ]:
# 7) Exportar + salvar no GitHub
from heilo.training.records import agora
destino = Path(BASE) / "heilo" / "models" / "seed_code" / "versions" / NOME_VERSAO
destino.mkdir(parents=True, exist_ok=True)
pretrain.exportar(modelo, destino / "heilo_seed_code.pt",
                  {"versao": NOME_VERSAO, "base": "v0.2", "passos": res["passo"], "hist": res["hist"],
                   "tarefas_verificadas": meta["relatorio"]})
manifesto = {"versao": NOME_VERSAO, "base": "v0.2", "criada": agora(), "tipo": "HEILO Faísca Code",
             "receita": "continuação da 0.2: 70% código público + 30% funções verificadas (pedido → solução testada)",
             "passos": res["passo"], "minutos": res["minutos"],
             "ultima_validacao": res["hist"][-1] if res["hist"] else None,
             "avaliacao": resumo, "comparacao_v0.2": {"pass@1": 0.0, "pass@6": 0.05, "codigo_valido": 0.93}}
(destino / "manifest.json").write_text(json.dumps(manifesto, indent=2, ensure_ascii=False))
(destino / "avaliacao.json").write_text(json.dumps(av, indent=2, ensure_ascii=False))
!cd {BASE}/heilo && git config user.email "heilo@colab" && git config user.name "HEILO (Colab)"
!cd {BASE}/heilo && git add -f models/seed_code/versions/{NOME_VERSAO} && git commit -q -m "HEILO Faísca Code {NOME_VERSAO} (Colab)" || echo nada
!cd {BASE}/heilo && git stash -u -q; git pull --rebase && git push && echo "ENVIADO PARA O GITHUB"